# RQ1: How does battery range vary with price across EV market segments?

**Research Question:** Is there a significant relationship between vehicle price and driving range, and does this relationship differ across market segments (Budget, Mid-range, Premium, Luxury)?

**Hypothesis:** Higher-priced EVs deliver significantly greater range, but with diminishing returns at premium price points, and this relationship differs significantly across market segments.

**Methodology:**
1. Load and clean the EV dataset
2. Compute descriptive statistics for price and range by market segment
3. Fit OLS regression: Range ~ Price (overall and per segment)
4. Compute Pearson correlation coefficients per segment
5. ANOVA test across segments
6. Generate scatter plot with regression lines per segment (saved as PDF)
7. Generate summary statistics table (saved as CSV)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from scipy.stats import f_oneway
import warnings
warnings.filterwarnings('ignore')

# Publication style
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'figure.dpi': 300,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print('Libraries loaded successfully.')

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────────
# On Kaggle the dataset is at /kaggle/input/<dataset-slug>/ev_market_2026.csv
# Adjust path if needed
import os
PATHS = [
    '/kaggle/input/electric-vehicle-market-and-pricing-dataset-2026/ev_market_2026.csv',
    'ev_market_2026.csv',
]
for p in PATHS:
    if os.path.exists(p):
        df = pd.read_csv(p)
        print(f'Loaded from: {p}')
        break

print(f'Shape: {df.shape}')
print(df[['price_usd', 'range_miles', 'market_segment']].describe())

In [ ]:
# ── Data cleaning ─────────────────────────────────────────────────────────────
df = df.dropna(subset=['price_usd', 'range_miles', 'market_segment'])
df['price_k'] = df['price_usd'] / 1000   # price in $1,000s for readability

SEGMENT_ORDER = ['Budget', 'Mid-range', 'Premium', 'Luxury']
COLORS = {'Budget': '#2166ac', 'Mid-range': '#4dac26', 'Premium': '#d6604d', 'Luxury': '#8b008b'}
MARKERS = {'Budget': 'o', 'Mid-range': 's', 'Premium': '^', 'Luxury': 'D'}

print('Segment counts:')
print(df['market_segment'].value_counts())

In [ ]:
# ── Statistical analysis ──────────────────────────────────────────────────────
results = []
for seg in SEGMENT_ORDER:
    sub = df[df['market_segment'] == seg]
    r, p = stats.pearsonr(sub['price_usd'], sub['range_miles'])
    slope, intercept, _, _, _ = stats.linregress(sub['price_usd'], sub['range_miles'])
    results.append({
        'Market Segment': seg,
        'N': len(sub),
        'Mean Price ($k)': round(sub['price_usd'].mean() / 1000, 1),
        'SD Price ($k)': round(sub['price_usd'].std() / 1000, 1),
        'Mean Range (mi)': round(sub['range_miles'].mean(), 1),
        'SD Range (mi)': round(sub['range_miles'].std(), 1),
        'Pearson r': round(r, 3),
        'p-value': round(p, 4),
        'Slope (mi/$k)': round(slope * 1000, 3),
    })

summary_df = pd.DataFrame(results)
print(summary_df.to_string(index=False))

# One-way ANOVA across segments
groups = [df[df['market_segment'] == s]['range_miles'].values for s in SEGMENT_ORDER]
F, p_anova = f_oneway(*groups)
print(f'\nOne-way ANOVA — F={F:.2f}, p={p_anova:.4f}')

In [ ]:
# ── Figure: Scatter + regression lines ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

for seg in SEGMENT_ORDER:
    sub = df[df['market_segment'] == seg]
    ax.scatter(
        sub['price_k'], sub['range_miles'],
        color=COLORS[seg], marker=MARKERS[seg],
        alpha=0.45, s=30, linewidths=0, label=seg
    )
    # Regression line
    slope, intercept, r, _, _ = stats.linregress(sub['price_k'], sub['range_miles'])
    x_range = np.linspace(sub['price_k'].min(), sub['price_k'].max(), 100)
    ax.plot(x_range, intercept + slope * x_range,
            color=COLORS[seg], linewidth=1.8, linestyle='--')
    # Annotate r value
    xpos = sub['price_k'].quantile(0.85)
    ypos = intercept + slope * xpos
    ax.annotate(f'r={r:.2f}', xy=(xpos, ypos),
                fontsize=8.5, color=COLORS[seg],
                xytext=(4, 4), textcoords='offset points')

ax.set_xlabel('Vehicle Price (USD $1,000s)', labelpad=8)
ax.set_ylabel('Driving Range (miles)', labelpad=8)
ax.set_title(
    'Price–Range Relationship Across EV Market Segments\n'
    f'(One-way ANOVA: F={F:.2f}, p={p_anova:.4f})',
    pad=12
)
ax.legend(title='Market Segment', frameon=True, framealpha=0.9, loc='upper left')

plt.tight_layout()
fig.savefig('RQ1_Price_Range_Scatter.pdf', bbox_inches='tight', format='pdf')
plt.show()
print('Figure saved: RQ1_Price_Range_Scatter.pdf')

In [ ]:
# ── Save summary table as CSV ─────────────────────────────────────────────────
summary_df.to_csv('RQ1_Summary_Table.csv', index=False)
print('Table saved: RQ1_Summary_Table.csv')
summary_df